In [ ]:
from games import Game, GameState, alpha_beta_search, random_player, query_player

class Hexapawn(Game):
    """
    เกม Hexapawn ขนาด 3x3 ตามมาตรฐาน AIMA
    - ผู้เล่น 'W' (ขาว): อยู่แถว 3 เดินขึ้นข้างบน (ทิศทาง -1)
    - ผู้เล่น 'B' (ดำ): อยู่แถว 1 เดินลงข้างล่าง (ทิศทาง +1)
    """
    def __init__(self, size=3):
        self.size = size
        board = {}
        # วางเบี้ยเริ่มต้น: ดำอยู่แถว 1, ขาวอยู่แถว 3
        for col in range(1, size + 1):
            board[(1, col)] = 'B'
            board[(size, col)] = 'W'
        
        initial_moves = self._get_moves(board, 'W')
        self.initial = GameState(to_move='W', utility=0, board=board, moves=initial_moves)

    def _get_moves(self, board, player):
        """คำนวณตาเดินที่ถูกกฎทั้งหมดของผู้เล่น player"""
        moves = []
        direction = -1 if player == 'W' else 1
        opp = 'B' if player == 'W' else 'W'
        
        for (r, c), p in board.items():
            if p == player:
                # 1. เดินหน้าตรง 1 ช่อง (ต้องเป็นช่องว่างเท่านั้น)
                fwd = (r + direction, c)
                if 1 <= fwd[0] <= self.size and fwd not in board:
                    moves.append(((r, c), fwd))
                
                # 2. กินเฉียงซ้าย 1 ช่อง (ต้องมีหมากคู่แข่ง)
                cap_left = (r + direction, c - 1)
                if 1 <= cap_left[0] <= self.size and 1 <= cap_left[1] <= self.size and board.get(cap_left) == opp:
                    moves.append(((r, c), cap_left))
                    
                # 3. กินเฉียงขวา 1 ช่อง (ต้องมีหมากคู่แข่ง)
                cap_right = (r + direction, c + 1)
                if 1 <= cap_right[0] <= self.size and 1 <= cap_right[1] <= self.size and board.get(cap_right) == opp:
                    moves.append(((r, c), cap_right))
        return moves

    def actions(self, state):
        """ส่งคืนตาเดินที่ทำได้จากสถานะปัจจุบัน"""
        return state.moves

    def result(self, state, move):
        """สร้างสถานะใหม่หลังจากทำการเดิน move"""
        from_pos, to_pos = move
        board = state.board.copy()
        del board[from_pos]
        board[to_pos] = state.to_move
        
        next_player = 'B' if state.to_move == 'W' else 'W'
        next_moves = self._get_moves(board, next_player)
        utility = self.compute_utility(board, move, state.to_move, next_moves)
        return GameState(to_move=next_player, utility=utility, board=board, moves=next_moves)

    def compute_utility(self, board, move, player, next_moves):
        """ตรวจสอบเงื่อนไขการชนะ 3 ข้อ"""
        from_pos, to_pos = move
        
        # เงื่อนไขที่ 1: เบี้ยเดินไปถึงแถวหลังสุดของคู่แข่ง
        if player == 'W' and to_pos[0] == 1:
            return 1   # ขาวชนะ
        if player == 'B' and to_pos[0] == self.size:
            return -1  # ดำชนะ
            
        # เงื่อนไขที่ 2: กินเบี้ยคู่แข่งจนหมดกระดาน
        opp = 'B' if player == 'W' else 'W'
        opp_count = sum(1 for p in board.values() if p == opp)
        if opp_count == 0:
            return 1 if player == 'W' else -1
            
        # เงื่อนไขที่ 3: ฝ่ายตรงข้ามไม่เหลือตาเดิน (โดนบล็อก / Stalemate)
        if len(next_moves) == 0:
            return 1 if player == 'W' else -1
            
        return 0

    def utility(self, state, player):
        """ส่งคืนค่าคะแนน (+1 ถ้าชนะ, -1 ถ้าแพ้, 0 ถ้ายังไม่จบ)"""
        return state.utility if player == 'W' else -state.utility

    def terminal_test(self, state):
        """เกมจบเมื่อมีคนชนะ หรือไม่เหลือตาเดิน"""
        return state.utility != 0 or len(state.moves) == 0

    def display(self, state):
        """แสดงผลกระดานหมากรุก 3x3"""
        print("  1 2 3")
        for r in range(1, self.size + 1):
            row_str = ' '.join(state.board.get((r, c), '.') for c in range(1, self.size + 1))
            print(f"{r} {row_str}")
        print(f"ตาของ: {state.to_move} | Utility: {state.utility}\n")


 
def alpha_beta_player(game, state):
    """ผู้เล่น AI ที่ใช้ Alpha-Beta Search คำนวณตาเดินที่ดีที่สุด"""
    return alpha_beta_search(state, game)



=== จำลองการแข่งขัน: Alpha-Beta (W) vs Random (B) ===
  1 2 3
1 B . B
2 W B W
3 . W .
ตาของ: B | Utility: 1

🏆 ผู้เล่น W (Alpha-Beta) ชนะ!


In [8]:
hex_game = Hexapawn()
print("=== จำลองการแข่งขัน: Alpha-Beta (W) vs Random (B) ===")
result = hex_game.play_game(alpha_beta_player, random_player)
if result == 1:
    print("🏆 ผู้เล่น W (Alpha-Beta) ชนะ!")
elif result == -1:
    print("🏆 ผู้เล่น B (Random) ชนะ!")

=== จำลองการแข่งขัน: Alpha-Beta (W) vs Random (B) ===
  1 2 3
1 W B .
2 . B B
3 . . W
ตาของ: B | Utility: 1

🏆 ผู้เล่น W (Alpha-Beta) ชนะ!


In [7]:
print("=== จำลองการแข่งขัน: Alpha-Beta (W) vs Alpha-Beta (B) ===")
result = hex_game.play_game(alpha_beta_player, alpha_beta_player)
print("ผลลัพธ์ (1 = ขาวชนะ, -1 = ดำชนะ):", result)
 

=== จำลองการแข่งขัน: Alpha-Beta (W) vs Alpha-Beta (B) ===
  1 2 3
1 . . B
2 . B W
3 B . .
ตาของ: W | Utility: -1

ผลลัพธ์ (1 = ขาวชนะ, -1 = ดำชนะ): -1
